In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df=pd.read_csv("final_crypto.csv")

In [ ]:
df=df[df["type"]!="XRP"]
df=df[df["type"]!="BTC"]

In [ ]:
train_coins = ['ETH', 'BNB', 'SOL', 'ADA']
test_coins = ['DOGE', 'AVAX']

train_df = df[df['type'].isin(train_coins)].copy()
test_df  = df[df['type'].isin(test_coins)].copy()

print("Coins in train:", train_df['type'].unique())
print("Coins in test:", test_df['type'].unique())

In [ ]:
df.columns.tolist()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder





drop_cols = [
    'label', 'date', 'type',
    'down_days_7',
    'taker_buy_volume',
    'z_score','taker_buy_ratio','taker_buy_ratio', 'negative_momentum',
]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df['label']

X_test = test_df.drop(columns=drop_cols)
y_test = test_df['label']


model = RandomForestClassifier(
    n_estimators=700,
    max_depth=13,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features='sqrt',
    criterion='entropy',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

model.fit(X_train, y_train)






test_preds = model.predict(X_test)
print("\n--- Test Results ---")
print(classification_report(y_test, test_preds, target_names=['Bigdown', 'Bigup','Stable']))


cm = confusion_matrix(y_test, test_preds)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['Bigdown', 'Bigup','Stable'],
                                     yticklabels=['Bigdown', 'Bigup','Stable'])
plt.title("Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()


feat_imp = pd.Series(model.feature_importances_, index=X_train.columns)
feat_imp.sort_values().plot(kind='barh', figsize=(8, 6))
plt.title("Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:

# Check train performance too
train_preds = model.predict(X_train)
print("--- Train Results ---")
print(classification_report(y_train, train_preds, target_names=['Bigdown', 'Bigup','Stable']))

print("--- Test Results ---")
print(classification_report(y_test, test_preds, target_names=['Bigdown', 'Bigup','Stable']))

In [ ]:
print(y_train.unique())
print(sorted(y_train.unique()))

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd
import numpy as np

drop_cols = [
    'label', 'date', 'type',
    'down_days_7',
    'taker_buy_volume',
    'z_score','taker_buy_ratio','taker_buy_ratio', 'negative_momentum',
]

X = df.drop(columns=drop_cols)
y = df['label']

groups = df['type']

logo = LeaveOneGroupOut()

results = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    test_coin = groups.iloc[test_idx].unique()[0]

    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=13,
        min_samples_split=5,
        min_samples_leaf=3,
        criterion='entropy',
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    macro_f1 = f1_score(y_test, preds, average='macro')
    weighted_f1 = f1_score(y_test, preds, average='weighted')

    results.append({
        'test_coin': test_coin,
        'accuracy': acc,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1
    })

    print(f"\n===== Fold {fold} | Test coin: {test_coin} =====")
    print(classification_report(y_test, preds))

In [ ]:
results_df = pd.DataFrame(results)

print(results_df)

print("\nAverage Results:")
print(results_df[['accuracy', 'macro_f1', 'weighted_f1']].mean())

# **SVM**

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
clf = SVC(kernel='rbf', gamma=0.01, C=1000, class_weight='balanced', random_state=42)
clf.fit(X_train_scaled, y_train)
predictions = clf.predict(X_test_scaled)

In [ ]:
svm_test_predictions = clf.predict(X_test_scaled)
svm_train_predictions = clf.predict(X_train_scaled)

print(f"Accuracy: {accuracy_score(y_test, svm_test_predictions):.4f}")
print("\n--- SVM Test Results ---")
print(classification_report(y_test, svm_test_predictions, target_names=['Bigdown', 'Bigup', 'Stable']))

cm_svm = confusion_matrix(y_test, svm_test_predictions)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Blues', xticklabels=['Bigdown', 'Bigup','Stable'], yticklabels=['Bigdown', 'Bigup','Stable'])
plt.title("SVM Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

print("\n--- SVM Train Results ---")
print(classification_report(y_train, svm_train_predictions, target_names=['Bigdown', 'Bigup', 'Stable']))

# **Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
print(f"Training Logistic Regression... Train: {len(X_train_scaled)}")
log_reg.fit(X_train_scaled, y_train)
log_predictions = log_reg.predict(X_test_scaled)

In [ ]:
log_test_predictions = log_reg.predict(X_test_scaled)
log_train_predictions = log_reg.predict(X_train_scaled)

print("\n--- Logistic Regression Test Results ---")
print(classification_report(y_test, log_test_predictions, target_names=['Bigdown', 'Bigup', 'Stable']))

cm_log = confusion_matrix(y_test, log_test_predictions)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_log, annot=True, fmt='d', cmap='Greens', xticklabels=['Bigdown', 'Bigup','Stable'], yticklabels=['Bigdown', 'Bigup','Stable'])
plt.title("Logistic Regression Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

importance = np.mean(np.abs(log_reg.coef_), axis=0)
feat_imp_log = pd.Series(importance, index=X_train.columns)
feat_imp_log.sort_values().plot(kind='barh', figsize=(8, 6), color='skyblue')
plt.title("Logistic Regression Feature Importance (Abs Coeff)")
plt.tight_layout()
plt.show()

print("--- Logistic Regression Train Results ---")
print(classification_report(y_train, log_train_predictions, target_names=['Bigdown', 'Bigup', 'Stable']))